<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_Pandas_02_Cleaning_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — Data Cleaning with Pandas

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this worksheet by going to **File > Save a copy in Drive**

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

**How today works.** Some code cells are blank on purpose:
- **type along** — a comment says what to write; type it with me, then run it.
- **predict first** — write your guess in the comment *before* you run the cell.
- **your turn** — try it on your own.

If a cell gives you an error, read the message — errors are part of the work here, not something to
hide.

Last class the data was *tidy*. Real found data almost never is. Scraped comments show up with
stray spaces, line breaks in the middle of text, the same category spelled five different ways, blank
cells, and duplicate rows. Before you can analyze anything, you have to **clean** it. Today you'll take
a messy version of the same comments table from last class and turn it back into something trustworthy —
and notice that every cleaning choice is also an *interpretive* choice.

## Why we're doing this

- **From the lecture.** *Data Archaeology* asked you to look at any whole-looking dataset and ask
  "which part is bone, and which part is plaster?" Cleaning is where you add the plaster. Every value
  you fill in, every label you merge, every row you drop is a choice you made, not something the
  commenters wrote.
- **Where you'll use it.** Friday's workshop is this notebook on a dataset *you* collect. HW3 runs
  sentiment on that cleaned data, and HW4 and your capstone build on it.
- **The question to keep asking.** If someone read only my cleaned table, what would they not know
  about the choices I made?

# A Realistically Messy Dataset

Here's what those tidy YouTube comments might actually look like fresh out of a scraper. Same columns
as last time — `comment`, `stance`, `likes`, `replies` — but full of real-world junk.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "comment": [
        "  The Ten Commandments belong in every classroom, period.  ",
        "This is a clear violation of church and state.\nKeep it out.",
        "Morals matter and kids today need them more than ever.",
        "Whose religion gets to decide? Not the government's job.",
        "Honestly I have no strong opinion either way.",
        "God and country, that's what built this nation.",
        "Public schools serve everyone, not just one faith.  ",
        "Put the Constitution in classrooms, not commandments.",
        "Finally some common sense values in our schools.",
        "Freedom of religion means freedom from it too.",
        "My kids should learn this at home, not at school.",
        "  The Ten Commandments belong in every classroom, period.  ",
    ],
    "stance": ["support", "Oppose", "SUPPORT", "against", "Neutral", "for",
               "oppose ", "OPPOSE", "support", "Against", "neutral", "support"],
    "likes": [240, 312, 88, 150, np.nan, 205, 176, 410, 64, 198, 33, 240],
    "replies": [15, 42, 6, 22, 1, 18, 19, 51, 4, 27, 2, 15],
}

messy_df = pd.DataFrame(data)
messy_df

# Step 1: Diagnose Before You Touch Anything

Good cleaning starts with *looking*. Three quick questions tell you most of what's wrong:
**What's missing? What's inconsistent? What's duplicated?**

**Predict first.** Look back at the table: which column has a missing value, and how many? Write your guess, then run.

In [ ]:
# my guess: column =    missing =
print("Missing values per column:")
print(messy_df.isnull().sum())

### What's inconsistent? `.unique()`

`.unique()` lists every distinct value in a column, once each. It's the quickest way to catch the same
category spelled several different ways.

**The structure:**
```python
my_df["column_name"].unique()
```
- `my_df["column_name"]` picks out one column, as in Pandas 01.
- `.unique()` gives back each different value once, in the order it first appears.
- Values that differ only in capital letters or a trailing space count as *different* values. That's
  exactly what you're hunting for.

In [ ]:
# type along — what's inconsistent? print every distinct value in the stance column with .unique()

### What's duplicated? `.duplicated()`

`.duplicated()` checks each row against the rows above it and marks it `True` if it's an exact copy of
one of them. Adding `.sum()` counts the `True`s. That's the same trick as `.isnull().sum()` above.

**The structure:**
```python
my_df.duplicated().sum()
```
- `.duplicated()` looks at the whole table, so there's no column name.
- A row counts as a duplicate only if *every* column matches, character for character, spaces included.
- The first copy isn't marked; only the repeats after it are.

In [ ]:
# type along — what's duplicated? count whole rows that repeat with .duplicated().sum()

So: one missing `likes` value, a `stance` column that means three things but is spelled ten
ways, and one repeated comment. Let's fix them one at a time on a **copy** (never destroy your raw
data — you may need to go back).

In [ ]:
clean_df = messy_df.copy()

# Step 2: Clean the Text Fields

Pandas applies a string method to a *whole column* at once through the `.str` accessor — it's the
column-wide version of the string methods you already know from *Strings & String Methods*. Strip the
whitespace, drop the stray newline.

**The structure:**
```python
my_df["column_name"] = my_df["column_name"].str.string_method()
```
- Put `.str` between the column and the method: `.str.strip()`, `.str.lower()`, `.str.replace(old, new)`.
  Any string method you know works this way.
- It runs on every row at once, so there's no loop to write.
- Chain several methods in a row, as below: each one works on the result of the one before it.
- It doesn't change the table by itself. Assign the result back to the column (`my_df["column_name"] = ...`)
  to keep it.
- `.tolist()` at the end prints the column as a plain Python list, so you can see the quotes and spot
  any leftover spaces.

One new piece: **`regex=False`** tells `.str.replace` to treat `"\n"` as plain text to find, character
for character. Without it, pandas may read the pattern as a *regular expression* (regex), a mini-language
where characters like `.` and `?` mean "any character" or "optional." You don't need regex in this course;
`regex=False` just says "find exactly this."

In [ ]:
clean_df["comment"] = clean_df["comment"].str.strip().str.replace("\n", " ", regex=False)
clean_df["comment"].tolist()

# Step 3: Standardize the Categories

`stance` is the worst offender. First flatten the casing and spacing, then **map** the synonyms
("for" → support, "against" → oppose) onto a clean set of three labels.

In [ ]:
# type along — chain .str.strip() and .str.lower() on the stance column, assign it back, then print .unique()

Now the labels are tidy, but "for" and "support" are still two words for one position. To merge
them, give `.replace()` a **dictionary** (from *Dictionaries & Functions*): each key is a label to
find, and its value is the label to put there instead.

**The structure:**
```python
my_df["column_name"] = my_df["column_name"].replace({"old_label": "new_label", "other_old": "other_new"})
```
- Unlike `.str.replace()`, this one has no `.str`. It swaps a **whole value** only when the entire
  cell matches, so `"for"` changes but `"forever"` wouldn't.
- Labels that aren't in the dictionary are left alone.
- Deciding that "for" means "support" is your call, not the commenter's. Note it in your writeup.

In [ ]:
# map the synonyms onto canonical labels
clean_df["stance"] = clean_df["stance"].replace({"for": "support", "against": "oppose"})
print("Canonical labels:", clean_df["stance"].unique())

### Your turn

In [ ]:
# your turn — run .value_counts() on the cleaned stance column to see the three-way split
# clean_df["stance"].value_counts()

# Step 4: Drop the Duplicate

The repeated comment you counted in Step 1 is an *exact* duplicate: every column matches, stray spaces
and all. `.drop_duplicates()` keeps the first copy and removes the rest.

**The structure:**
```python
my_df = my_df.drop_duplicates().reset_index(drop=True)
```
- `.drop_duplicates()` removes every row that `.duplicated()` marked `True`.
- Dropping rows leaves a gap in the row numbers on the left (the *index*). `.reset_index(drop=True)`
  renumbers them 0, 1, 2… and throws the old numbers away.
- We do this *before* Step 5 on purpose: a comment counted twice would also count twice in any
  average or median you take.

In [ ]:
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(clean_df)}")
clean_df

# Step 5: Handle the Missing Value

One comment has no `likes` count. You have choices, and **none are neutral**:
- **Drop** the row (`.dropna()`) — but you lose a comment (in real data, someone's voice).
- **Fill** it (`.fillna()`) with an estimate — but you're inventing data. This is the *plaster*.

Here we fill with the **median**: the middle value when the numbers are sorted. One very popular
comment (like the 410) pulls the mean up, but it barely moves the median. Whatever you choose, *say so*
in your writeup — it's part of your method.

**The structure:**
```python
my_value = my_df["column_name"].median()
my_df["column_name"] = my_df["column_name"].fillna(my_value)
```
- `.median()` works like `.mean()` from Pandas 01: it gives back one number for the column.
- `.fillna(my_value)` puts that value into every blank cell in the column and leaves the rest alone.
- Like the `.str` methods, it doesn't change the table until you assign it back.

In [ ]:
# type along — save the median of the likes column as `median_likes`; fill the missing value with .fillna(); print the median and how many are still missing

# Putting It All Together: Clean Data Tells a Truer Story

The whole point of cleaning is that the analysis you couldn't trust before, you can trust now. With
the labels standardized and the duplicate gone, the counts finally mean something.

In [ ]:
print("Stance breakdown (clean):")
print(clean_df["stance"].value_counts())

print("\nAverage likes by stance:")
print(clean_df.groupby("stance")["likes"].mean().round(1))

Before cleaning, `.value_counts()` on `stance` would have shown *ten* labels for what are really
three positions, plus a comment counted twice — a garbage picture. Same data, cleaned, and the crowd's
split comes through.

You can also now reliably search the text. This puts together the filter from Pandas 01
(`my_df[condition]`) with the `.str` accessor:

**The structure:**
```python
my_df[my_df["column_name"].str.lower().str.contains("word")]
```
- `.str.contains("word")` gives `True` for every row whose text includes that word anywhere.
- Running `.str.lower()` first means "Religion" and "religion" both match.
- The outer `my_df[ ... ]` keeps only the `True` rows, just like `comments_df[comments_df["likes"] > 150]`.

In [ ]:
# which comments invoke "religion" directly?
religion = clean_df[clean_df["comment"].str.lower().str.contains("religion")]
religion[["comment", "stance"]]

In [ ]:
# your turn — filter to comments mentioning "schools" (or another word) and see who's on which side

# Sneak Preview: Where This Is Going

**This Friday's workshop** is you doing exactly this — to a dataset *you* collected. Diagnose it,
standardize it, drop the junk, until it's analysis-ready. Cleaning is unglamorous, but it's where
honest analysis begins: every choice you make here (what to fill, what to drop, how to label) quietly
shapes the conclusions you'll draw later.

And next week the payoff arrives: with a clean `comment` column, **VADER** will score each comment's
*tone*: how positive or negative its words are. That is a **second, separate measure**. It does not
replace the "support / oppose / neutral" stance you labeled by hand; a comment can oppose the law in
cheerful words or support it angrily. Keep both columns side by side, because the places where they
disagree are some of the most interesting findings you'll make. Clean data in, real analysis out.

Further out, your **capstone** stands on whatever cleaning you do now. Its notebook has a
"clean, and say what you changed" step, where you note what you filled, merged, and dropped. Keep
notes as you go, and the plaster stays visible.

In [ ]:
# a peek at next week: clean text is what sentiment scoring needs
clean_df[["comment", "stance"]].head()

# Playground

Try one of these, or anything else you're curious about:

- **Drop instead of fill.** Start again from `messy_df.copy()` and use `.dropna()` instead of filling
  the blank. Rerun the "Average likes by stance" line. Did any side's average move? Which version
  would you report, and why?
- **Argue with the map.** Is "neutral" really one position, or could "Honestly I have no strong
  opinion" and "My kids should learn this at home" deserve different labels? Write the dictionary
  you'd use instead, run it, and see how `.value_counts()` changes.
- **Hunt for a near-duplicate.** Add a row to `data` that's the same comment with one extra space or
  a capital letter. Does `.duplicated()` catch it before cleaning? After Step 2?

In [ ]:
# use this space to experiment!